# 04 - Comparaison de quelques images


Comparer les signatures HSV et LBP de plusieurs images, puis classer les resultats. Cette experience porte sur un petit echantillon et ne constitue pas encore un index complet.

**Préparer les dépendances de la comparaison**

- La cellule `%pip` installe les mêmes dépendances qu’au notebook 03 dans le noyau courant. La comparaison recalcule les descripteurs du petit échantillon, donc elle a besoin de Pillow et de scikit-image même si un autre notebook a déjà affiché des signatures.
- NumPy calcule les histogrammes et les distances. Matplotlib ouvre la figure de requête et la galerie finale. Chaque fichier de notebook peut être exécuté depuis son propre noyau.

In [ ]:
%pip install pillow numpy matplotlib scikit-image

**Extraire les signatures et définir la distance**

- `signatures(chemin)` redimensionne l’image en 256 × 256, calcule le HSV conjoint avec huit cases par canal et le LBP uniforme avec huit voisins et un rayon de 1. La fonction retourne deux vecteurs distincts, de longueurs 512 et 10. Chaque vecteur est divisé par son propre total.
- Pour comparer deux histogrammes, `np.sqrt(a * b).sum()` additionne leur recouvrement case par case. Le résultat vaut 1 quand deux distributions sont identiques et 0 quand elles n’ont aucune case occupée en commun. La distance de Hellinger prend la racine de **1 moins ce recouvrement**.
- `np.clip` ramène l’expression dans l’intervalle [0, 1] avant la racine, ce qui évite notamment une très petite valeur négative due aux arrondis. Une distance basse indique des distributions similaires selon le descripteur choisi.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from skimage.feature import local_binary_pattern

base = Path("data/101_ObjectCategories")
if not base.is_dir():
    raise FileNotFoundError(f"Dossier introuvable : {base.resolve()}. Ouvre le notebook depuis la racine du projet.")

# Memes parametres que dans le notebook 03.
def signatures(chemin):
    with Image.open(chemin) as image:
        rgb = image.convert("RGB").resize((256, 256), Image.Resampling.LANCZOS)
    hsv = np.asarray(rgb.convert("HSV"))
    gris = np.asarray(rgb.convert("L"))
    couleur, _ = np.histogramdd(hsv.reshape(-1, 3), bins=(8, 8, 8),
                               range=((0, 256), (0, 256), (0, 256)))
    lbp = local_binary_pattern(gris, 8, 1, method="uniform")
    texture, _ = np.histogram(lbp, bins=np.arange(11))
    return (couleur / couleur.sum()).ravel(), texture / texture.sum()

# Distance de Hellinger, aussi appelee distance de Bhattacharyya dans OpenCV.
def distance_histogrammes(a, b):
    return np.sqrt(np.clip(1.0 - np.sqrt(a * b).sum(), 0.0, 1.0))

## Choisir une requete et un echantillon


On prend une image de requete et quelques images issues de plusieurs categories. Les chemins peuvent etre modifies pour explorer d autres exemples.

**Choisir un échantillon et contrôler la requête**

- Le code prend les trois premiers JPEG triés dans chacun des quatre dossiers. Une des douze images est la requête et le filtre la retire. L’exemple par défaut compare donc **11 candidats** à une image d’avion.
- Les assertions contrôlent la longueur et la somme des deux vecteurs de requête. La comparaison de chacun avec lui-même doit donner une distance proche de zéro. La tolérance numérique accepte les petites erreurs de calcul.
- La figure montre la photo à retrouver avant le classement. Le dossier `airplanes` indique sa catégorie et reste utile pour lire les résultats. Ce nom ne participe pas au calcul des distances.

In [ ]:
requete = base / "airplanes" / "image_0001.jpg"
categories = ["airplanes", "Motorbikes", "Faces", "watch"]
if not requete.is_file():
    raise FileNotFoundError(f"Requete introuvable : {requete}")
candidats = []
for categorie in categories:
    images = sorted((base / categorie).glob("*.jpg"))
    if len(images) < 3:
        raise ValueError(f"Moins de trois images dans la categorie : {categorie}")
    candidats.extend(chemin for chemin in images[:3] if chemin != requete)

couleur_requete, texture_requete = signatures(requete)
assert couleur_requete.shape == (512,) and texture_requete.shape == (10,)
assert np.isclose(couleur_requete.sum(), 1) and np.isclose(texture_requete.sum(), 1)
assert np.isclose(distance_histogrammes(couleur_requete, couleur_requete), 0, atol=1e-7)
assert np.isclose(distance_histogrammes(texture_requete, texture_requete), 0, atol=1e-7)
print("Requete :", requete, "| candidats :", len(candidats))
with Image.open(requete) as image:
    plt.imshow(image.convert("RGB"))
    plt.axis("off")
    plt.title("Image de requete")
    plt.show()

## Calculer les distances et classer


Les deux distances sont comprises entre 0 et 1 : plus elles sont petites, plus les histogrammes se ressemblent. Le score combine utilise des poids initiaux egaux ; ils restent a evaluer.

**Fusionner les distances et lire le classement**

- Pour chaque candidat, on calcule séparément une distance HSV et une distance LBP. Le score vaut `poids_couleur * d_couleur + (1 - poids_couleur) * d_texture`. Avec 0,5, les deux distances contribuent à parts égales. Avec 1, seule la couleur intervient. Avec 0, seule la texture intervient.
- Le tri croissant utilise le premier élément de chaque tuple, le score combiné. La sortie texte permet de voir si un candidat est retenu surtout pour sa couleur ou pour sa texture. La galerie affiche jusqu’à cinq candidats dans le même ordre.
- Essaie de changer le poids et de relancer cette cellule. Observe les images dont le rang change. Le petit échantillon sert à comprendre le classement. L’évaluation sur plusieurs catégories sera nécessaire pour comparer les méthodes.

In [ ]:
poids_couleur = 0.5
assert 0 <= poids_couleur <= 1
resultats = []
for chemin in candidats:
    couleur, texture = signatures(chemin)
    assert couleur.shape == couleur_requete.shape and texture.shape == texture_requete.shape
    assert np.isclose(couleur.sum(), 1) and np.isclose(texture.sum(), 1)
    d_couleur = distance_histogrammes(couleur_requete, couleur)
    d_texture = distance_histogrammes(texture_requete, texture)
    score = poids_couleur * d_couleur + (1 - poids_couleur) * d_texture
    resultats.append((score, d_couleur, d_texture, chemin))

resultats.sort(key=lambda resultat: resultat[0])
for score, d_couleur, d_texture, chemin in resultats:
    print(f"{chemin.parent.name:12} {chemin.name:16} couleur={d_couleur:.3f} texture={d_texture:.3f} combine={score:.3f}")

fig, axes = plt.subplots(1, min(5, len(resultats)), figsize=(15, 3))
for ax, (score, d_couleur, d_texture, chemin) in zip(np.atleast_1d(axes), resultats[:5]):
    with Image.open(chemin) as image:
        ax.imshow(image.convert("RGB"))
    ax.set_title(f"{chemin.parent.name}\nscore {score:.3f}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Interpre­ter les resultats


Essaie plusieurs requetes et modifie poids_couleur entre 0 et 1 : 1 utilise seulement la couleur, 0 seulement la texture. Les categories ne garantissent pas une ressemblance visuelle ; le faible echantillon ne permet pas de conclure sur la performance. La comparaison euclidienne et le choix des poids restent a etudier.